# House Sales Price Prediction — King County, WA

**Role:** Data Analyst at a residential real estate investment trust, evaluating whether acquisition targets are priced fairly.

**Question:** what does square footage, location, and condition tell us about what a house *should* sell for — and how much better can we do with a full model than with any single feature?


## About the Dataset

House sale prices for King County (which includes Seattle), covering homes sold between May 2014 and May 2015. Originally from [Kaggle](https://www.kaggle.com/harlfoxem/housesalesprediction); lightly modified for teaching purposes (two columns have some values deliberately blanked out, handled in the wrangling step below).


| Variable      | Description                                                                                                 |
| ------------- | ----------------------------------------------------------------------------------------------------------- |
| id            | A notation for a house                                                                                      |
| date          | Date house was sold                                                                                         |
| price         | Price is prediction target                                                                                  |
| bedrooms      | Number of bedrooms                                                                                          |
| bathrooms     | Number of bathrooms                                                                                         |
| sqft_living   | Square footage of the home                                                                                  |
| sqft_lot      | Square footage of the lot                                                                                   |
| floors        | Total floors (levels) in house                                                                              |
| waterfront    | House which has a view to a waterfront                                                                      |
| view          | Has been viewed                                                                                             |
| condition     | How good the condition is overall                                                                           |
| grade         | Overall grade given to the housing unit, based on King County grading system                                |
| sqft_above    | Square footage of house apart from basement                                                                 |
| sqft_basement | Square footage of the basement                                                                              |
| yr_built      | Built Year                                                                                                  |
| yr_renovated  | Year when house was renovated                                                                               |
| zipcode       | Zip code                                                                                                    |
| lat           | Latitude coordinate                                                                                         |
| long          | Longitude coordinate                                                                                        |
| sqft_living15 | Living room area in 2015 (implies some renovations); might not affect lot size area                         |
| sqft_lot15    | Lot size area in 2015 (implies some renovations)                                                            |


## Setup


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split, cross_val_score

%matplotlib inline

## 1. Loading the Data

The dataset is hosted publicly, so it loads straight into a dataframe with `pandas.read_csv()` — no manual download step needed.


In [ ]:
file_name = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DA0101EN-SkillsNetwork/labs/FinalModule_Coursera/data/kc_house_data_NaN.csv'
df = pd.read_csv(file_name)
df.head()

## 2. Data Wrangling

### Are the data types sensible?

Before doing anything else, check what pandas inferred for each column.


In [ ]:
df.dtypes

`id` and `zipcode` load as integers even though they're really categorical identifiers, and `date` loads as a plain string rather than a real datetime. None of that breaks the analysis below, but it's worth flagging — `id` in particular needs to be dropped rather than treated as a numeric feature.


### Dropping non-predictive identifiers

`id` and the stray `Unnamed: 0` index column carry no predictive signal, so drop them before going any further. Re-running `describe()` afterward both confirms the drop worked and gives a first look at the range of every remaining numeric column.


In [ ]:
df.drop(['id', 'Unnamed: 0'], axis=1, inplace=True)
df.describe()

Price is heavily right-skewed, as home prices almost always are — a relatively small number of high-end sales pull the mean well above the median. That skew is worth keeping in mind later: a plain linear regression will still fit, but a handful of expensive outliers can pull its coefficients around more than a typical sale would justify.


### Handling missing values

`bedrooms` and `bathrooms` have some missing values, deliberately introduced for this exercise.


In [ ]:
print("number of NaN values for the column bedrooms :", df['bedrooms'].isnull().sum())
print("number of NaN values for the column bathrooms :", df['bathrooms'].isnull().sum())

Mean imputation is the simplest fix — it keeps every row in the dataset rather than throwing sales away — but it also quietly shrinks the variance of these two columns toward the average. That's a reasonable trade-off here since only a small fraction of rows are affected, but it's worth naming if a stakeholder ever asks why bedroom count doesn't correlate with price as strongly as they'd expect.


In [ ]:
mean=df['bedrooms'].mean()
df['bedrooms'].replace(np.nan,mean, inplace=True)

In [ ]:
mean=df['bathrooms'].mean()
df['bathrooms'].replace(np.nan,mean, inplace=True)

In [ ]:
print("number of NaN values for the column bedrooms :", df['bedrooms'].isnull().sum())
print("number of NaN values for the column bathrooms :", df['bathrooms'].isnull().sum())

## 3. Exploratory Data Analysis

### How common is each floor count?


In [ ]:
floor_counts = df['floors'].value_counts().to_frame()
floor_counts

Most homes in the county are 1 or 1.5-story — worth remembering when interpreting the `floors` coefficient in the models below. A feature that barely varies across the dataset can't carry much predictive signal, no matter how intuitive it seems that more floors should mean a bigger, pricier house.


### Does a waterfront view command a price premium?


In [ ]:
sns.boxplot(x='waterfront', y='price', data=df)

Waterfront homes are rare in this dataset, and boxplots for a small, rare category often show a wide spread with a few extreme high-end outliers. Read the *median* line as the more reliable signal here, rather than the highest point in the box — with so few waterfront sales, one unusually expensive property can dominate the top of the range.


### Does more above-ground square footage mean a higher price?


In [ ]:
sns.regplot(x='sqft_above', y='price', data=df)
plt.ylim(0,)

Expect a clear positive slope — above-ground living space is one of the most consistent price drivers in residential real estate. At the highest end the relationship typically flattens out a bit, as location and lot characteristics start to matter more than raw size.


### Ranking every numeric feature by its correlation with price


In [ ]:
df_numeric = df.select_dtypes(include=[np.number])
df_numeric.corr()['price'].sort_values()

In most published analyses of this dataset, `sqft_living` and `grade` come out as the strongest numeric correlates with price, with `bedrooms` noticeably weaker than most buyers would guess. Check whether that pattern holds here — and note anything that surprises you, since a counter-intuitive correlation is often the most useful thing to flag for a stakeholder.


## 4. Model Development

### Baseline: can longitude alone predict price?

Start with a deliberately weak single-feature model, as a floor to compare everything else against.


In [ ]:
X = df[['long']]
Y = df['price']
lm = LinearRegression()
lm.fit(X,Y)
lm.score(X, Y)

Expect a low R² here — a raw longitude coordinate is a weak predictor on its own. Location clearly matters for home price, but not as a single continuous number; it likely needs to be combined with latitude, or treated more like a categorical zip code, to be useful.


### Does square footage alone do better than location alone?


In [ ]:
X = df[['sqft_living']]
Y = df['price']
lm = LinearRegression()
lm.fit(X, Y)
lm.score(X, Y)

Compare this R² against the longitude-only baseline above — expect it to be dramatically higher. Living area is a far stronger single predictor than a raw coordinate, which is exactly why square footage is the number buyers ask for first.


### Combining features: how much better is a multi-feature model?

Bring in floor count, waterfront status, location, room counts, and grade together.


In [ ]:
features =["floors", "waterfront","lat" ,"bedrooms" ,"sqft_basement" ,"view" ,"bathrooms","sqft_living15","sqft_above","grade","sqft_living"]     

In [ ]:
X = df[features]
Y = df['price']
lm = LinearRegression()
lm.fit(X, Y)
lm.score(X, Y)

This should outperform every single-feature model above — it's drawing on size, location, and quality together rather than any one signal alone. The R² gap between this and the `sqft_living`-only model is a good number to have ready when explaining to a stakeholder why analysts build multi-feature models instead of leaning on one "obvious" variable.


### Adding non-linearity: does a scaled polynomial pipeline help?

Wrap feature scaling, second-degree polynomial expansion, and linear regression into a single pipeline.


In [ ]:
Input=[('scale',StandardScaler()),('polynomial', PolynomialFeatures(include_bias=False)),('model',LinearRegression())]

In [ ]:
pipe = Pipeline(Input)
pipe.fit(df[features], df['price'])
pipe.score(df[features], df['price'])

Polynomial terms let the model capture non-linear effects — like an extra bedroom mattering more in a small house than a huge one. But this score is still measured in-sample. Watch for a large gap between this number and the train/test results in the next section — that gap is what overfitting looks like.


## 5. Model Evaluation & Refinement

Every R² so far has been measured on the same data the model was trained on — which flatters any model, especially the polynomial one above. From here on, evaluate on a held-out test set instead.


In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
print("done")

In [ ]:
features =["floors", "waterfront","lat" ,"bedrooms" ,"sqft_basement" ,"view" ,"bathrooms","sqft_living15","sqft_above","grade","sqft_living"]    
X = df[features]
Y = df['price']

x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.15, random_state=1)


print("number of test samples:", x_test.shape[0])
print("number of training samples:",x_train.shape[0])

### Does regularization change the picture on unseen data?


In [ ]:
from sklearn.linear_model import Ridge

In [ ]:
RidgeModel = Ridge(alpha=0.1)
RidgeModel.fit(x_train, y_train)
RidgeModel.score(x_test, y_test)

This is the first model scored on genuinely held-out data rather than in-sample fit. Expect the R² to land somewhat below the plain multi-feature regression's in-sample score from Section 4 — that drop is normal, and it's exactly why train/test evaluation matters more than in-sample fit when the goal is a model that generalizes to houses it hasn't seen.


### Combining regularization and non-linearity


In [ ]:
pr = PolynomialFeatures(degree=2)
x_train_pr = pr.fit_transform(x_train)
x_test_pr = pr.fit_transform(x_test)

RidgeModel = Ridge(alpha=0.1)
RidgeModel.fit(x_train_pr, y_train)
RidgeModel.score(x_test_pr, y_test)

Compare this test R² against the plain Ridge model just above. If it's meaningfully higher, the polynomial terms are adding genuine predictive power on unseen data, not just fitting noise. If it's about the same or lower, the extra complexity isn't earning its keep here, and the simpler model would be the better one to ship.


## Key Findings & Recommendation

*(Fill in with your actual results after running the notebook — the exact R² values will differ slightly depending on the `train_test_split` random state.)*

- **Best single feature:** `sqft_living` vs. the longitude baseline (Section 4) — how much does swapping one variable for a better one change R²?
- **Multi-feature lift:** the 11-feature model vs. `sqft_living` alone — how much more does combining size, location, and grade explain?
- **Generalization check:** the in-sample pipeline score (Section 4) vs. the test-set Ridge scores (Section 5) — how big is the overfitting gap, and does the polynomial Ridge model close it or widen it?

**Recommendation for the Trust:** report the test-set R² and leading coefficients from the best-generalizing model above, and call out which 2–3 features (typically living area, grade, and location) matter most when evaluating acquisition targets.


---
*Dataset and question set adapted from IBM's "Data Analysis with Python" course (House Sales in King County, USA). Analysis and code completed independently.*
